# Shaft Connections, exercise 3 <br>

<div>
<img src="images/Lab_PressFit2.PNG" width="350"/>
</div>

On a E335 drive shaft (diameter $d=50mm$) a pinion (Dutch: rondsel) with helical gears (material: 16MnCr5) is to be dry-crimped. The pion's pitch circle (Dutch: steekcirkel) is 91mm and the root circle (Dutch: voetcirkel) is 84.65mm.  <br>
- The transmissible power $P=20kW$
- rotational speed $n=560 min^-1$
<br> Including the adverse effects of starting/stopping the forces can be assumed to be:
- tangential force $F_T = 13,1kN$
- Axial force $F_a = 2.3 kN$
- radial force $F_R = 4.8 kN$
- contact surfaces can be assumed to have a roughness $R_z = 6.3 \mu m$
- in order to avoid stress concentrations, the edge of the inner hole of the pinion are rounded with $L_e = 4mm$
<br>

calculate, using the basic hole system, the appropriate ISO fitting to be used. Assume average factors.
<br>
At which temperature do you have to heat the outer part? Assume an ambient temperature of 20°C



### setting correct lib

In [1]:
# import statements
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt

import MechDesign.Helpers as HM

from MechDesign.Units.Units import m_, mm_, kg_, s_, N_, rpm_, W_, deg_, mu_m_
import MechDesign.Units.UnitMethods as UM

import MechDesign.RnM as RnM

### straight forward calculation for a single, specific shaft connection
given specific values for all variables, the average surface pressure $ p_{gem} $ is calculated

### determine minimum surface pressure needed

In [2]:
# creating a new shaft connection, (this is 1 specific connection in your design)
from sympy import N


SC = RnM.ShaftConnection() 

# setting up constants, based on assignment (units are specified using a trailing '_')
SC.D_Ii = 0 * mm_
SC.D_F = 50 *mm_
SC.D_Uu = 84.65 *mm_
D_steek = 91.4 *mm_
SC.Rz_Iu = 6.3 * 1e-6 *m_
SC.Rz_UI = 6.3 * 1e-6 *m_
SC.l_F = (80-2*4) *mm_
SC.mu = 0.19

F_T = 13.1*1e3*N_
F_A = 2.3*1e3*N_
F_R = 4.8*1e3*N_

SC.K_A = 1 # can be set to 1, the text states that K_A is already taken into account in the F values

# setting up constants that are not likely to change
SC.S_S = 1.75

# setting up helper functions
SC.F_t = F_T*D_steek/SC.D_F
SC.F_l = F_A

SC.F_res = SC.E12_8_hA_F_res()
SC.F_res = sp.powdenest(SC.F_res,force=True)  # used to force sqrt(N²) to be N
SC.F_S = SC.E12_8_F_S()
HM.EqPrint('F_S',SC.F_S)

SC.A_F = SC.E12_9_hA_AF()
HM.EqPrint('A_F',SC.A_F.evalf())

SC.p_Fmin = SC.E12_9_p_Fmin()
t=HM.EqPrint('p_Fmin',SC.p_Fmin.evalf())



Eq(F_S, 42100.0*N_)

Eq(A_F, 11310.0*mm_**2)

Eq(p_Fmin, 19.59*N_/mm_**2)

### Determine minimum surplus size
fist determine the value of the helper value $K$

In [3]:
# some more constants
SC.E_U = 210*1e3 *N_/mm_/mm_
SC.E_I = 210*1e3 *N_/mm_/mm_
SC.nu_U = 0.3
SC.nu_I = 0.3

# helper functions
SC.Q_U = SC.E12_5_hA_Q_U()
HM.EqPrint('Q_U',SC.Q_U)
SC.Q_I = SC.E12_5_hB_Q_I()
HM.EqPrint('Q_I',SC.Q_I)


# main function
SC.K = SC.E12_12_K()
t=HM.EqPrint('K',SC.K)

Eq(Q_U, 0.5907)

Eq(Q_I, 0)

Eq(K, 3.072)

### Minimum oversize


In [4]:
# minimum absolute press fit
SC.Z_min = SC.E12_13_Z_min()
HM.EqPrint('Z_min',SC.Z_min.evalf())   # adding the ".evalf()" converts the 'pi' symbol to the number 3.1415...

# roughness reduction
SC.G = SC.E12_14_G()
HM.EqPrint('G',SC.G)

# minimum absolute press fit
SC.S_nmin = SC.E12_15_S_nmin()
SC.S_nmin = SC.S_nmin.evalf(subs={m_:1e6*mu_m_,mm_:1e3*mu_m_})
t=HM.EqPrint('S_nmin',SC.S_nmin)


Eq(Z_min, 0.01433*mm_)

Eq(G, 1.008e-5*m_)

Eq(S_nmin, 24.41*mu_m_)

### maximum oversize

In [5]:
# setting safety 
SC.S_pI = 1.15
SC.S_pU = 1.15
SC.R_eU = 695*N_/mm_/mm_*SC.K_t
SC.R_eI = 335*N_/mm_/mm_*SC.K_t


# creating new, not yet defined symbols
K_tU = 0.92
K_tI = 0.96

# using expression E_12_16
p_FmaxU = SC.E12_16A_p_Fmax_Outer()
p_FmaxU = p_FmaxU.evalf(subs={SC.K_t:K_tU})
HM.EqPrint('p_FmaxU',p_FmaxU)

p_FmaxI = SC.E12_16B_p_Fmax_InnerMassive()
p_FmaxI = p_FmaxI.evalf(subs={SC.K_t:K_tI})
t=HM.EqPrint('p_FmaxI',p_FmaxI)

Eq(p_FmaxU, 209.0*N_/mm_**2)

Eq(p_FmaxI, 322.9*N_/mm_**2)

the outer part is the limiting factor

In [6]:
# since the outer part is the limiting factor
SC.p_Fmax = p_FmaxU

# calculating the maximum absolute press fitt
SC.Z_max = SC.E12_17_Z_max()

HM.EqPrint('Z_max',SC.Z_max)

# the maximum allowable over dimension becomes
SC.S_nmax = SC.E12_18_Snmax()
HM.EqPrint('S_nmax',SC.S_nmax)
SC.S_nmax = UM.m_to_mu_m(UM.All_to_SI(SC.S_nmax)) 
HM.EqPrint('S_nmax',SC.S_nmax)
t=HM.EqPrint('S_nmin',SC.S_nmin)



Eq(Z_max, 0.1529*mm_)

Eq(S_nmax, 1.008e-5*m_ + 0.1529*mm_)

Eq(S_nmax, 162.9*mu_m_)

Eq(S_nmin, 24.41*mu_m_)

### determining the fitting


In [7]:
# overall permissible total tolerance 
SC.P_T =SC.E12_19_P_T()
HM.EqPrint('P_T',SC.P_T)
# assigning a portion of the tolerance to T_B
SC.T_B = SC.E12_20_hA_TB()
t=HM.EqPrint('T_B',SC.T_B)

Eq(P_T, 138.5*mu_m_)

Eq(T_B, 83.12*mu_m_)

H9 on a diameter 50 returns, $62\mu m$ 
<br>
but when using H9, only 40% should be attributed to $T_B$

In [8]:
SC.T_B = 0.4*SC.P_T
t=HM.EqPrint('T_B',SC.T_B)

Eq(T_B, 55.41*mu_m_)

the next smaller IT class now becomes H8 -> 36µm

In [9]:
# determine lower bound
P_asMin = 36 * mu_m_ + SC.S_nmin
HM.EqPrint('P_asMin',P_asMin)
# next possible value: u = 70 µm
# since Sn_max = 163 µm 
P_asMax = (163-70)*mu_m_
t=HM.EqPrint('P_asMax',P_asMax)
# P_as = 93 µm
# next smaller tolerance class for diameter 50: class6-> 16 *µm


Eq(P_asMin, 60.41*mu_m_)

Eq(P_asMax, 93*mu_m_)

the overall fitting to be used could b e H9u6 <br>
this, however, would be a strange combination of a very rough hole with a precise shaft. This would also not stick to boundary conditions for use of the 40%60% split of the fit. 
For now, let's keep it and look at mounting temperatures.

### mounting temperature outer component

In [10]:
# setting up some constants

SC.T =  20
SC.T_I = 20
SC.alpha_I = 8.5*1e-6
SC.alpha_U = 11*1e-6
SC.S_m = SC.S_nmax/2
SC.T_room = 20

SC.Sprime_nmax = 86*mu_m_  #86 µm = maximum interferenc using H9u6

# main expression
SC.T_U = SC.E12_22_T_U_TemperatureOuterPart()
SC.T_U = SC.T_U.evalf(subs = {mm_:1e3*mu_m_})
t=HM.EqPrint('T_U',SC.T_U)


Eq(T_U, 324.5)

this is above the advised temperature of 250°C.  Therefore we need to cool the shaft.

In [11]:
# suppose we would could the shaft to -250°C
SC.T_I = 20-(325-250)
# main expression
SC.T_U = SC.E12_22_T_U_TemperatureOuterPart()
SC.T_U = SC.T_U.evalf(subs = {mm_:1e3*mu_m_})
t=HM.EqPrint('T_U',SC.T_U)


Eq(T_U, 266.5)

This would still not satisfy requirements and would be very impractical, the shaft would also be very brittle at those temperatures
<br>
although using class 9 might satisfy press fit conditions, it is rather large and is the cause of the excessive need for heating and cooling.  Switching to a higher tollerance class will yield better results. 
<br><br>
e.g: starting from a H7, which is still not to hard to manufacture 
<br>
diamter 50, H7:  0 to +25µm

In [12]:
# minimum size of the shaft
P_asMin = 25*mu_m_ + SC.S_nmin
HM.EqPrint('P_asMin',P_asMin)
# next step over is t, with 54 µm starting point
# maximum diameter of shaft is depends on p_Fmax and corresponding S_nmax
P_asMax = SC.S_nmax - 54*mu_m_
t=HM.EqPrint('T_U',P_asMax)

Eq(P_asMin, 49.41*mu_m_)

Eq(T_U, 108.9*mu_m_)

using e.g. H8/t7, the maximum interference would become 79µm
<br>
this would however mean we, again, have to revert back to the 60%40% split.  
<br>
the main focus is to find a fit that works, preferably a standard fit. 


In [13]:
SC.Sprime_nmax = 79*mu_m_  #79 µm = maximum interference using H8/t7

# main expression
SC.T_U = SC.E12_22_T_U_TemperatureOuterPart()
SC.T_U = SC.T_U.evalf(subs = {mm_:1e3*mu_m_})
t=HM.EqPrint('T_U',SC.T_U)

Eq(T_U, 253.8)

Heating temperature becomes 253°C, without the need for cooling the shaft. 